# 03 — Fusion Validation

Phase 2b evaluation notebook. Covers six sections:

1. **Strategy recovery** — fused vs single-modality baselines
2. **Ablation** — leave-one-out modality importance
3. **Geometry** — UMAP of CDT embeddings (archetype + latent param)
4. **Cross-modal retrieval** — participant-level nearest-neighbour recall
5. **PersonaConfig regression probe** — latent parameter recovery R²
6. **Text encoder diagnostic** — intra-archetype cosine similarity

In [ ]:
import sys
import warnings

sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
import os

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "1"

import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from schemas import PERSONA_LABELS, CHECKPOINT_PATHS
from fusion.meta_learner import LateFusionMetaLearner

cache = torch.load("models/fusion_embeddings_cache.pt", weights_only=False)
labels = cache["labels"].numpy()
MODALITIES = ["trace", "transaction", "text", "psychographic"]
print(f"Cache loaded: {len(labels)} participants, {len(np.unique(labels))} archetypes")

## 1. Strategy Recovery

The >85% overall accuracy is the only hard gate (fusion/SPEC.md Tier 1). Text and psychographic each achieve 100% individually because they are near-sufficient statistics for PersonaConfig. Fusion not exceeding them is expected — fusing two 100% classifiers cannot improve beyond 100%.

In [ ]:
from evaluation.strategy_recovery import run_strategy_recovery, format_results

results = run_strategy_recovery()
print(format_results(results))

In [ ]:
BASELINES = {"trace": 0.9502, "transaction": 0.6259, "text": 1.0, "psychographic": 1.0}
modalities = ["trace", "transaction", "text", "psychographic", "fusion"]
accs = [
    BASELINES["trace"],
    BASELINES["transaction"],
    BASELINES["text"],
    BASELINES["psychographic"],
    results["val_accuracy"],
]
colors = ["#4C72B0", "#4C72B0", "#4C72B0", "#4C72B0", "#DD8452"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(modalities, accs, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(0.85, color="red", linestyle="--", linewidth=1.5, label="85% gate")
ax.set_ylim(0, 1.10)
ax.set_ylabel("Val accuracy")
ax.set_title("Strategy Recovery: Single-modality vs Fused")
ax.legend()
for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{acc:.0%}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
plt.tight_layout()
plt.savefig("notebooks/strategy_recovery.png", dpi=150)
plt.show()

## 2. Ablation — Leave-one-out Modality Importance

Each modality is ablated by zeroing its 128-dim slice in the fusion input. The accuracy delta measures how much the fused model relies on that modality.

**Expected:** Low delta for text/psychographic because they are redundant with each other — both encode PersonaConfig directly. Trace has the largest delta because it is the only signal the model cannot reconstruct from the other three.

In [ ]:
from evaluation.ablation import run_ablation, format_results as ablation_fmt

ablation = run_ablation()
print(ablation_fmt(ablation))

In [ ]:
deltas = {
    m: ablation["baseline_accuracy"] - ablation["ablation_results"][m]["val_accuracy"]
    for m in MODALITIES
}
sorted_mods = sorted(deltas, key=lambda m: -deltas[m])
delta_vals = [deltas[m] for m in sorted_mods]
bar_colors = ["#c0392b" if d >= 0.05 else "#e8a87c" for d in delta_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(sorted_mods, delta_vals, color=bar_colors, edgecolor="white")
ax.axvline(0.05, color="gray", linestyle="--", linewidth=1, label="5% reference")
ax.set_xlabel("Accuracy drop when modality removed")
ax.set_title("Leave-one-out Modality Importance")
ax.legend()
for i, (m, d) in enumerate(zip(sorted_mods, delta_vals)):
    ax.text(d + 0.001, i, f"{d:.1%}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("notebooks/ablation_importance.png", dpi=150)
plt.show()

## 3. Geometry — UMAP of CDT Embeddings

**(a)** Coloured by archetype — tests between-persona separation.

**(b)** Coloured by `price_sensitivity` — tests within-persona variation preservation.

If clusters in (b) show a gradient, the CDT preserves continuous latent variation. If the gradient is flat within clusters, the model has collapsed to a pure archetype classifier with no within-archetype geometry.

In [ ]:
from evaluation.geometry import compute_umap

model = LateFusionMetaLearner()
model.load_state_dict(torch.load("models/fusion_meta_learner.pt", weights_only=True))
model.eval()

emb_list = [F.normalize(cache[m], dim=-1) for m in MODALITIES]
fusion_input = torch.cat(emb_list, dim=-1)
with torch.no_grad():
    cdt_embs = model.embed(fusion_input).numpy()

reducer, embedding_2d = compute_umap(cdt_embs)
print(f"UMAP shape: {embedding_2d.shape}")

In [ ]:
# (a) Coloured by archetype
palette = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(9, 6))
for idx, label in enumerate(PERSONA_LABELS):
    mask = labels == idx
    ax.scatter(
        embedding_2d[mask, 0],
        embedding_2d[mask, 1],
        c=[palette[idx]],
        label=label,
        s=12,
        alpha=0.7,
    )
ax.legend(fontsize=8, markerscale=2)
ax.set_title("CDT Embeddings — UMAP coloured by archetype")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("notebooks/umap_archetype.png", dpi=150)
plt.show()

In [ ]:
# (b) Coloured by price_sensitivity
configs = {}
with open("data/synthetic/participant_configs.jsonl") as f:
    for line in f:
        r = json.loads(line)
        configs[r["participant_id"]] = r

participant_ids = cache["participant_ids"]
price_sens = np.array(
    [configs.get(pid, {}).get("price_sensitivity", 0.5) for pid in participant_ids]
)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1],
    c=price_sens,
    cmap="RdYlGn_r",
    s=12,
    alpha=0.7,
    vmin=0,
    vmax=1,
)
plt.colorbar(sc, ax=ax, label="price_sensitivity")
ax.set_title("CDT Embeddings — UMAP coloured by price_sensitivity")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("notebooks/umap_price_sensitivity.png", dpi=150)
plt.show()
print(
    "Gradient within clusters -> CDT preserves within-archetype variation.\n"
    "Uniform colour within clusters -> embedding collapsed to archetype-only."
)

## 4. Cross-modal Retrieval

For each participant, we use their CDT (or single-modality) embedding as a query and search for the same participant in a different modality's embedding space by cosine nearest-neighbour.

**Results: recall@1 ≈ 0.001–0.003 (all tests). Within-archetype chance baseline: 1/143 ≈ 0.007.**

**Interpretation:** The CDT retrieves the correct individual at near-zero rate — below random. This confirms the latent variable recovery framing: the CDT has collapsed to archetype-level identity and cannot distinguish individuals within an archetype. This is consistent with the training objective (7-class archetype classification, not metric learning on individuals). Section 5 (config_probe) is the more sensitive test of whether any continuous latent variation survives.

In [ ]:
from evaluation.retrieval import evaluate as retrieval_eval

ret = retrieval_eval(log_mlflow=False)

print(f"  {'Modality':<15} recall@1   recall@10")
print(f"  {'-' * 40}")
for mod, m in ret["cdt_vs_single"].items():
    print(f"  {mod:<15} {m['recall_at_1']:.4f}     {m['recall_at_10']:.4f}")
print()
print("Single-vs-single:")
for pair, m in ret["single_vs_single"].items():
    print(f"  {pair:<32} {m['recall_at_1']:.4f}")
print(f"\nWithin-archetype chance baseline: {1 / 143:.4f}")

## 5. PersonaConfig Regression Probe

Ridge regression (α=1.0) predicts each PersonaConfig float parameter from each embedding space. R² reported on the val split (same 80/20 split, seed=42 as fusion training).

**Key findings:**
- Fused embedding achieves the highest R² on every parameter — fusion integrates complementary information
- `inspection_depth` shows the largest fusion gain: fused 0.982 vs trace 0.863
- Transaction encoder R² near-zero or negative — consistent with its 62.59% archetype accuracy
- Despite near-zero individual retrieval (Section 4), the CDT encodes continuous latent variation (R² ≥ 0.73 for all params) — the signal is there, just not at individual resolution

In [ ]:
from evaluation.config_probe import probe

probe_results = probe(log_mlflow=False)

CONFIG_PARAMS = list(probe_results.keys())
PROBE_MODALITIES = ["fused", "trace", "transaction", "text", "psychographic"]
r2_matrix = np.array(
    [[probe_results[p][m] for m in PROBE_MODALITIES] for p in CONFIG_PARAMS]
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    r2_matrix,
    annot=True,
    fmt=".3f",
    xticklabels=PROBE_MODALITIES,
    yticklabels=CONFIG_PARAMS,
    cmap="YlGn",
    vmin=-0.1,
    vmax=1.0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "R²"},
)
ax.set_title("PersonaConfig Regression Probe — R² by parameter × modality")
ax.set_xlabel("Embedding space")
plt.tight_layout()
plt.savefig("notebooks/config_probe_heatmap.png", dpi=150)
plt.show()

## 6. Text Encoder Diagnostic

The text encoder achieves 100% archetype classification. Hypothesis: the sentence-transformer separates persona narratives trivially in LLM space — the narratives are stereotyped enough that any LLM would cluster them by archetype.

We test this by computing mean intra-archetype cosine similarity of raw sentence-transformer embeddings (before the trained projection head). If mean similarity > 0.95, the narratives are trivially separable.

In [ ]:
from encoders.text.embed import TextEncoder

text_enc = TextEncoder(n_classes=7)
text_state = torch.load(CHECKPOINT_PATHS["text"], weights_only=True)
text_enc.load_state_dict(text_state, strict=False)
text_enc.eval()

narratives = {}
with open("data/synthetic/narratives.jsonl") as f:
    for line in f:
        r = json.loads(line)
        narratives[r["participant_id"]] = r.get("text", "")

participant_ids = cache["participant_ids"]
texts = [narratives.get(pid, "") for pid in participant_ids]
valid_mask = [bool(t) for t in texts]
texts_valid = [t for t, v in zip(texts, valid_mask) if v]
labels_valid = labels[[i for i, v in enumerate(valid_mask) if v]]

with torch.no_grad():
    sent_embs = text_enc.encode_texts(texts_valid)
sent_norm = F.normalize(sent_embs, dim=-1)

print(f"  {'Archetype':<25} mean_intra_sim  n")
print(f"  {'-' * 50}")
for idx, label in enumerate(PERSONA_LABELS):
    mask = labels_valid == idx
    if mask.sum() < 2:
        continue
    embs_arch = sent_norm[mask]
    sim_mat = (embs_arch @ embs_arch.T).numpy()
    np.fill_diagonal(sim_mat, np.nan)
    print(f"  {label:<25} {np.nanmean(sim_mat):.4f}          {mask.sum()}")
print("\nThreshold: mean_sim > 0.95 -> trivially separable in LLM space")

## Summary

| Section | Finding | Status |
|---|---|---|
| Strategy recovery | 100% val_acc | ✓ Tier 1 PASS (>85% gate) |
| Ablation | Trace: 10.4% delta; text/psych: 0% | ✓ Expected (redundancy, not failure) |
| UMAP geometry | See plots above | Qualitative |
| Individual retrieval | recall@1 ≈ 0.001–0.003 | CDT collapsed to archetype identity |
| Config probe | Fused R² best on all 7 params | ✓ Fusion integrates complementary signal |
| Text diagnostic | See intra-sim values | Interpret against 100% text accuracy |

**Core finding:** The CDT embedding is a high-quality archetype classifier and a moderate latent-param regressor, but not an individual-level retrieval system. This is consistent with the training objective and the latent variable recovery framing documented in project-vision.md.